In [2]:
!python scripts/generate_dataset.py

In [5]:
import json
train_data = []
with open('data/train.jsonl', 'r') as f:
    for line in f.readlines():
        if line.strip():
            train_data.append(json.loads(line))

print(train_data[0])

In [12]:
import os
os.environ.keys()
os.environ["GOOGLE_CLOUD_PROJECT"] = os.environ["ANTHROPIC_VERTEX_PROJECT_ID"]
os.environ["GOOGLE_CLOUD_REGION"] = os.environ["CLOUD_ML_REGION"]

In [13]:
from api_adapter.api_client import get_async_client, DEFAULT_MODEL, SYSTEM_PROMPT


client = get_async_client()

message = await client.messages.create(
    model=DEFAULT_MODEL,
    system=SYSTEM_PROMPT,
    messages=[
        {"role": "user", "content": f"Evaluate: 25 - 14"}
    ],
    max_tokens=100,
)
message.content

In [14]:

!python scripts/run_baseline.py

In [18]:
!uv pip install -e ".[gpu,dev]"

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python scripts/train_grpo.py --condition D --max-steps 1000

In [24]:
!CUDA_VISIBLE_DEVICES=0 uv run scripts/train_grpo.py --condition D --max-steps 1000 --resume outputs/grpo_condition_D/checkpoint-800

In [26]:
!CUDA_VISIBLE_DEVICES=0 python scripts/analyze_condition_d.py --model-path outputs/grpo_condition_D/final-adapter

In [30]:
!CUDA_VISIBLE_DEVICES=0 python scripts/analyze_condition_d.py

In [32]:
test_predictions = []
with open('outputs/grpo_condition_D/test_predictions.jsonl', 'r') as f:
    for line in f.readlines():
        if line.strip():
            test_predictions.append(json.loads(line))

test_predictions[0]



In [34]:
# split into standard and non-standard
standard_predictions = []
non_standard_predictions = []
for prediction in test_predictions:
    if prediction['type'] == 'standard':
        standard_predictions.append(prediction)
    else:
        non_standard_predictions.append(prediction)

non_standard_predictions[0]

In [35]:
import random
random.choice(non_standard_predictions)

## Use Only the trained model directly.

In [36]:
import json
import unsloth  # noqa: F401
from unsloth import FastLanguageModel
from vllm import SamplingParams

from api_adapter.api_client import SYSTEM_PROMPT as API_SYSTEM_PROMPT
from api_adapter.evaluate import evaluate_predictions
from api_adapter.reward import extract_answer
from api_adapter.symbols import SYMBOL_DEFINITIONS_VAGUE

adapter_path = "outputs/grpo_condition_D/final_adapter"
test_data_path = "data/baseline/test_baseline.jsonl"

with open(test_data_path) as f:
    test_items = [json.loads(line) for line in f if line.strip()]

test_items[0]

In [37]:
print(f"Loading adapter from {adapter_path}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=adapter_path,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=False,
    fast_inference=True,
    gpu_memory_utilization=0.5,
)
FastLanguageModel.for_inference(model)


In [76]:
def build_user_prompt_no_fewshot(item):
    claude_answer = item.get("claude_answer")
    claude_answer_str = str(claude_answer) if claude_answer is not None else "none"
    return "\n".join([
        f"Expression: {item['expression']} ",
        "/no_think",
    ])

SYSTEM_PROMPT = "You are an arithmetic calculator. Evaluate the given expression and return the result."
messages_list = [
    [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt_no_fewshot(item)},
    ]
    for item in test_items
]

messages_list[0]

In [77]:
texts = [
    tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    for messages in messages_list
]

sampling_params = SamplingParams(
    temperature=1.0,
    max_tokens=1024,
)

outputs = model.fast_generate(
    texts,
    sampling_params=sampling_params,
    lora_request=model.load_lora(adapter_path),
)


In [ ]:
predictions_api_system_no_fewshot = []
for item, output in zip(test_items, outputs):
    raw_text = output.outputs[0].text
    predicted = extract_answer(raw_text, claude_answer=item.get("claude_answer"))
    predictions_api_system_no_fewshot.append({
        "expression": item["expression"],
        "answer": item["answer"],
        "claude_answer": item.get("claude_answer"),
        "predicted": predicted,
        "type": item["type"],
        "output": raw_text,
    })

for x in predictions_api_system_no_fewshot:
    if x['type'] == 'custom':
        display(x)
        break

In [79]:

metrics_api_system_no_fewshot = evaluate_predictions(
    predictions_api_system_no_fewshot,
    label="Ablation 1: API system prompt + adapter model + no few-shot",
)
metrics_api_system_no_fewshot

#### Ablation 2:

In [70]:
from api_adapter.symbols import SYMBOL_DEFINITIONS_VAGUE

def build_user_prompt_fewshot(item):
    expression = item['expression']
    claude_answer = item['claude_answer']
    parts = []
    parts.append(SYMBOL_DEFINITIONS_VAGUE)
    parts.append("Examples:")
    parts.append("Expression: 3 θ 4 → \\boxed{7}")
    parts.append("Expression: 10 α 3 → \\boxed{7}")
    parts.append("Expression: 2 γ 6 → \\boxed{12}")
    parts.append("")

    parts.append(f"Expression: {expression} →")
    parts.append("/no_think")
    return "\n".join(parts)

build_user_prompt_fewshot(test_items[0])

In [71]:
SYSTEM_PROMPT = "You are an arithmetic calculator. Evaluate the given expression and return the result."

messages_list = [
    [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt_fewshot(item)},
    ]
    for item in test_items
]

messages_list[0]

In [73]:
texts = [
    tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    for messages in messages_list
]

sampling_params = SamplingParams(
    temperature=1.0,
    max_tokens=1024,
)

outputs = model.fast_generate(
    texts,
    sampling_params=sampling_params,
    lora_request=model.load_lora(adapter_path),
)


In [74]:
predictions_api_system_fewshot = []
for item, output in zip(test_items, outputs):
    raw_text = output.outputs[0].text
    predicted = extract_answer(raw_text, claude_answer=item.get("claude_answer"))
    predictions_api_system_fewshot.append({
        "expression": item["expression"],
        "answer": item["answer"],
        "claude_answer": item.get("claude_answer"),
        "predicted": predicted,
        "type": item["type"],
        "output": raw_text,
    })

for x in predictions_api_system_fewshot:
    if x['type'] == 'custom':
        display(x)
        break

In [75]:
metrics_api_system_fewshot = evaluate_predictions(
    predictions_api_system_fewshot,
    label="Ablation 2: API system + adapter few-shot",
)
metrics_api_system_fewshot

#### Ablation 3: re-shuffle the symbol mapping

Earlier:
- θ: +
- α: -
- γ: x
- β: /


New:
- θ: x
- α: /
- γ: -
- β: +

In [ ]:
# update test item expressions.
# find the indices of characters θ, α, γ, β
# create a new string with the characters replaced.

for x in test_items:
    if x['type'] == 'custom':
        break

x['expression']

In [62]:
REMAPPING_DICT = {
    'θ': 'β',
    'α': 'γ',
    'γ': 'θ',
    'β': 'α',
}

x['expression'].translate(str.maketrans(REMAPPING_DICT))


In [63]:
import copy
ablation_3_test_items = copy.deepcopy(test_items)
for x in ablation_3_test_items:
    if x['type'] == 'custom':
        x['expression'] = x['expression'].translate(str.maketrans(REMAPPING_DICT))

ablation_3_test_items[0]

In [64]:
from api_adapter.symbols import SYMBOL_DEFINITIONS_VAGUE

def build_user_prompt_fewshot_shuffled(item):
    expression = item['expression']
    claude_answer = item['claude_answer']
    parts = []
    parts.append(SYMBOL_DEFINITIONS_VAGUE)
    parts.append("Examples:")
    parts.append("Expression: 3 β 4 → \\boxed{7}")
    parts.append("Expression: 10 γ 3 → \\boxed{7}")
    parts.append("Expression: 2 θ 6 → \\boxed{12}")
    parts.append("")

    parts.append(f"Expression: {expression} →")
    parts.append("/no_think")
    return "\n".join(parts)

build_user_prompt_fewshot_shuffled(test_items[0])

In [65]:
SYSTEM_PROMPT = "You are an arithmetic calculator. Evaluate the given expression and return the result."

messages_list = [
    [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt_fewshot_shuffled(item)},
    ]
    for item in ablation_3_test_items
]

messages_list[0]

In [69]:
texts = [
    tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    for messages in messages_list
]

sampling_params = SamplingParams(
    temperature=1.0,
    max_tokens=1024,
)

outputs = model.fast_generate(
    texts,
    sampling_params=sampling_params,
    lora_request=model.load_lora(adapter_path),
)

predictions_api_system_fewshot_shuffled = []
for item, output in zip(ablation_3_test_items, outputs):
    raw_text = output.outputs[0].text
    predicted = extract_answer(raw_text, claude_answer=item.get("claude_answer"))
    predictions_api_system_fewshot_shuffled.append({
        "expression": item["expression"],
        "answer": item["answer"],
        "claude_answer": item.get("claude_answer"),
        "predicted": predicted,
        "type": item["type"],
        "output": raw_text,
    })

metrics_api_system_fewshot_shuffled = evaluate_predictions(
    predictions_api_system_fewshot_shuffled,
    label="Ablation 3: API system + adapter few-shot (shuffled)",
)
metrics_api_system_fewshot_shuffled

In [68]:
for x in predictions_api_system_fewshot_shuffled:
    if x['type'] == 'custom':
        display(x)
        break